# Phase 1H — Hands-on Exercise: Build a Memory System

**Task:** build a memory system for a customer-support agent using **two techniques** from different
families. Test it on a sample conversation. Report what it remembers, what it forgets, and where it
would fail.

**Independent notebook** — runs standalone in Colab or locally.

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "experiment" / "github_submission" / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

from embedder import TfidfHashEmbedder
from llm_client import LLMConfig, make_client, OfflineLLMClient
from memory_core import MemoryRecord
from providers import build_provider
emb = TfidfHashEmbedder()
try:
    llm = make_client(LLMConfig(backend='openai-compatible' if os.environ.get('OPENAI_API_KEY') else 'offline',
                                base_url=os.environ.get('OPENAI_BASE_URL','https://api.openai.com/v1'),
                                model='gpt-4o'))
except Exception:
    llm = OfflineLLMClient()
print('LLM backend:', llm.backend, '| API key:', bool(os.environ.get('OPENAI_API_KEY')))
TURNS = [
 ("Alice", "Hi, I am Alice. I work as a data scientist at a health-tech startup in Berlin."),
 ("Bob", "I am Bob, an ML engineer in Athens. I prefer PyTorch."),
 ("Alice", "We deploy on Kubernetes and track runs with Weights and Biases."),
 ("Bob", "Our training run failed last night with CUDA OOM at batch 256."),
 ("Alice", "We hit that before. Reducing batch to 64 and enabling gradient checkpointing fixed it."),
 ("Bob", "Our best val_loss was 0.423 with lr=3e-4 and weight_decay=0.01."),
 ("Alice", "I live in Prenzlauer Berg. My favorite coffee shop is on Kollwitzplatz."),
 ("Bob", "Let us sync next Tuesday at 10am CET."),
]
records = [MemoryRecord(f"t{i}", f"{w}: {t}", {"session_id": "s1" if i < 4 else "s2"}) for i, (w, t) in enumerate(TURNS)]
print("setup OK | turns =", len(TURNS))


## The scenario

A customer-support agent for a SaaS product. Customers report issues across multiple sessions.
The agent needs to **remember** who said what, **retrieve** relevant context when a customer
returns, and **forget** irrelevant details to stay efficient.

Here's a sample multi-session conversation:

In [ ]:
# Sample customer-support conversation (3 sessions)
SUPPORT_TURNS = [
 ('customer','Hi, I am Sarah from Acme Corp. Our API keeps returning 500 errors.'),
 ('agent','Hello Sarah. Can you share the endpoint and the error message?'),
 ('customer','POST /api/v1/orders returns 500. The error says database connection timeout.'),
 ('agent','That sounds like a connection pool issue. Let me check your account settings.'),
 ('customer','Thanks. Our account ID is ACME-42.'),
 # --- session 2 (next day) ---
 ('customer','Hi again, this is Sarah from Acme Corp. The 500 error is back.'),
 ('agent','Welcome back Sarah. I see your previous ticket about POST /api/v1/orders.'),
 ('customer','Yes, but now it also affects GET /api/v1/products. Same timeout error.'),
 ('agent','The connection pool issue may have spread. Let me escalate this.'),
 # --- session 3 (a week later) ---
 ('customer','This is Sarah from Acme. Is the connection pool issue resolved?'),
 ('agent','Let me check your history, Sarah.'),
]
records = [MemoryRecord(f'r{i}', f'{who}: {text}', {'session_id': f's{(i//4)+1}'})
 for i, (who, text) in enumerate(SUPPORT_TURNS)]
print(f'{len(records)} turns across {len(set(r.metadata["session_id"] for r in records))} sessions')

## Your task

1. **Choose two techniques** from different families (e.g., `verbatim` + `episodic`, or
 `extracted_facts` + `hybrid`). Think about what the support agent needs:
 - Remember the customer's identity and account (entity/fact memory).
 - Recall the specific error and endpoint (verbatim/vector).
 - Summarize the resolution across sessions (episodic/summary).

2. **Run both** on the sample conversation using the cells below.

3. **Answer:** what does each technique remember? What does it forget? Where would it fail?
 (e.g., does it recall Sarah's account ID? Does it remember the connection pool diagnosis?)

In [ ]:
# Technique 1 — your choice (change the name)
TECHNIQUE_1 = 'verbatim' # try: verbatim, extracted_facts, episodic, hybrid
p1 = build_provider(TECHNIQUE_1, emb, llm)
p1.ingest(records)
q1 = 'What is Sarah account ID and what error is she experiencing?'
items1, _ = p1.retrieve(q1, top_k=3)
print(f'{TECHNIQUE_1}: memory_size={p1.size()}')
for i, item in enumerate(items1):
 print(f' [{i}] score={item.score:.3f} | {item.content[:70]}')

In [ ]:
# Technique 2 — your choice (different family)
TECHNIQUE_2 = 'episodic' # try a different one
p2 = build_provider(TECHNIQUE_2, emb, llm)
p2.ingest(records)
q2 = 'What was the diagnosis for the connection pool issue?'
items2, _ = p2.retrieve(q2, top_k=3)
print(f'{TECHNIQUE_2}: memory_size={p2.size()}')
for i, item in enumerate(items2):
 print(f' [{i}] score={item.score:.3f} | {item.content[:70]}')

## Report (fill in)

| | Technique 1 | Technique 2 |
|---|---|---|
| What it remembers well | ... | ... |
| What it forgets | ... | ... |
| Where it would fail | ... | ... |

**Discuss with your neighbor:** which combination would you deploy for a real support agent?
What would you add (e.g., entity extraction for customer profiles)?

## Bridge to Phase 2

You've built memory systems and seen where they fail. **Phase 2** asks: how do we *measure*
those failures rigorously? The 3-probe diagnostic framework (relevance, utilization, failure
root-cause) gives you the tools to quantify exactly what your memory system gets right and wrong
on real benchmarks. Open `tutorial/phase2_public_datasets/` next.